In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

from util.db_helpers import get_mysql_engine

#---------------------------------------------
# Config
#---------------------------------------------

# Set to True to force re-downloading tables from MySQL even if a cached
# parquet file already exists in output/pipeline.
FORCE_MYSQL_REFRESH = False
MYSQL_DB = '2023_parcel_baseyear'

In [2]:
# download tables from MySQL and cache them as parquet files in output/pipeline
output_dir = Path('output/pipeline')
output_dir.mkdir(parents=True, exist_ok=True)

table_names = ['parcels', 'buildings', 'growth_centers','households','jobs','building_types','land_use_types']
dataframes = {}
engine = None

for table in table_names:
    parquet_path = output_dir / f'{table}.parquet'
    if parquet_path.exists() and not FORCE_MYSQL_REFRESH:
        dataframes[table] = pd.read_parquet(parquet_path)
    else:
        if engine is None:
            engine = get_mysql_engine(MYSQL_DB)
        df = pd.read_sql_table(table, engine)
        df.to_parquet(parquet_path)
        dataframes[table] = df

parcels_df = dataframes['parcels']
buildings_df = dataframes['buildings']
growth_centers_df = dataframes['growth_centers']
households_df = dataframes['households']
jobs_df = dataframes['jobs']
building_types_df = dataframes['building_types']
land_use_types_df = dataframes['land_use_types']

In [3]:
land_use_include = [
    1,	#agriculture
    2,	#Civic and Quasi-Public
    3,	#Commercial
    4,	#Fisheries
    5,	#Forest, harvestable
    6,	#Forest, protected
    7,	#Government
    8,	#Group Quarters
    9,	#Hospital, Convalescent Center
    10,	#Industrial
    11,	#Military
    12,	#Mining
    13,	#Mobile Home Park
    14,	#Multi-Family Residential
    15,	#Condo Residential
    16,	#n/a
    17,	#No Land Use Code
    18,	#Office
    19,	#Park and Open Space
    20,	#Parking
    21,	#Recreation
    22,	#Right-of-Way
    23,	#School
    24,	#Single Family Residential
    25,	#Transportation, Communication, Utilities
    26,	#Vacant Developable
    27,	#Vacant Undevelopable
    28,	#Warehousing
    29,	#Water
    30,	#Mixed Use
]

In [4]:
land_use_res = [
    8,	#Group Quarters
    13,	#Mobile Home Park
    14,	#Multi-Family Residential
    15,	#Condo Residential
    24,	#Single Family Residential
]

In [5]:
parcels_df['is_residential'] = parcels_df['land_use_type_id'].isin(land_use_res).astype(int)

In [6]:
parcel_mask = (parcels_df['land_use_type_id'].isin(land_use_include)) & (parcels_df['growth_center_id'] > 0)
parcel_sqft = parcels_df.loc[parcel_mask].groupby(['county_id','growth_center_id','is_residential'])['parcel_sqft'].sum().reset_index()
parcel_sqft['is_residential'] = parcel_sqft['is_residential'].map({1: 'res', 0: 'non_res'})
parcel_sqft = parcel_sqft.pivot_table(index=['county_id','growth_center_id'], columns='is_residential', values=['parcel_sqft'], aggfunc='sum')
parcel_sqft.columns = [f'{col}_{suffix}' for col, suffix in parcel_sqft.columns]
parcel_sqft = parcel_sqft.reset_index()
parcel_sqft['parcel_sqft_all'] = parcel_sqft['parcel_sqft_res'] + parcel_sqft['parcel_sqft_non_res']
for col in ['parcel_sqft_res', 'parcel_sqft_non_res', 'parcel_sqft_all']:
    parcel_sqft[col] = parcel_sqft[col].round(0).fillna(0).astype(int)

In [7]:
# add is_residential and growth_center_id columns to buildings_df
is_residential = building_types_df[['building_type_id','is_residential']].set_index('building_type_id')['is_residential']
buildings_df['is_residential'] = buildings_df['building_type_id'].map(is_residential)
parcel_center_xwalk = parcels_df[['parcel_id','growth_center_id']].set_index('parcel_id')['growth_center_id']
buildings_df['growth_center_id'] = buildings_df['parcel_id'].map(parcel_center_xwalk)
county_xwalk = parcels_df[['parcel_id','county_id']].set_index('parcel_id')['county_id']
buildings_df['county_id'] = buildings_df['parcel_id'].map(county_xwalk)

In [8]:
far = buildings_df.loc[buildings_df['growth_center_id'] != 0].groupby(['county_id','growth_center_id','is_residential'])[['gross_sqft']].sum().reset_index()
far['is_residential'] = far['is_residential'].map({1: 'res', 0: 'non_res'})
far = far.pivot_table(index=['county_id','growth_center_id'], columns='is_residential', values=['gross_sqft'], aggfunc='sum')
far.columns = [f'{col}_{suffix}' for col, suffix in far.columns]
far = far.reset_index()
far = far.merge(parcel_sqft, on=['county_id','growth_center_id'], how='left')

In [9]:
far['gross_sqft_all'] = far['gross_sqft_res'] + far['gross_sqft_non_res']
for type in ['res','non_res','all']:
    far[f'far_{type}'] = far[f'gross_sqft_{type}'] / far[f'parcel_sqft_{type}']

In [10]:
building_parcel_xwalk = buildings_df[['building_id','parcel_id']].set_index('building_id')['parcel_id']
households_df['parcel_id'] = households_df['building_id'].map(building_parcel_xwalk)
households_df['growth_center_id'] = households_df['parcel_id'].map(parcel_center_xwalk)
households_df['county_id'] = households_df['parcel_id'].map(county_xwalk)
persons = households_df.loc[households_df['growth_center_id'] != 0].groupby(['county_id','growth_center_id'])[['persons']].sum().reset_index()

In [11]:
jobs_df['parcel_id'] = jobs_df['building_id'].map(building_parcel_xwalk)
jobs_df['growth_center_id'] = jobs_df['parcel_id'].map(parcel_center_xwalk)
jobs_df['county_id'] = jobs_df['parcel_id'].map(county_xwalk)
jobs_mask = (jobs_df['growth_center_id'] != 0) & (jobs_df['home_based_status'] == 0)
jobs = jobs_df.loc[jobs_mask].groupby(['county_id','growth_center_id']).size().reset_index(name='jobs')

In [12]:
au = persons.merge(jobs, on=['county_id','growth_center_id'], how='outer').merge(far, on=['county_id','growth_center_id'], how='outer')
au['activity_units'] = au['persons'] + au['jobs']
au['acres_land'] = au['parcel_sqft_all'] / 43560
au['au_acre'] = au['activity_units'] / au['acres_land']

In [13]:
county_map = {
    33: 'King',
    35: 'Kitsap',
    53: 'Pierce',
    61: 'Snohomish'
}
au['county'] = au['county_id'].map(county_map)

growth_centers = growth_centers_df[['growth_center_id','name']].set_index('growth_center_id')['name']
au['growth_center'] = au['growth_center_id'].map(growth_centers)

# Chart

In [14]:
#| title: "Activity Units per Acre vs. FAR"
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook_connected"

valid = au[['county', 'growth_center', 'far_all', 'au_acre']].dropna()

slope, intercept = np.polyfit(valid['far_all'], valid['au_acre'], 1)
x_line = np.linspace(valid['far_all'].min(), valid['far_all'].max(), 100)

fig = px.scatter(
    valid,
    x='far_all',
    y='au_acre',
    hover_name='growth_center',
    hover_data={'county': True, 'far_all': ':.2f', 'au_acre': ':.1f'},
    labels={'far_all': 'FAR', 'au_acre': 'Activity Units / Acre'},
)
# Assign (rather than call as a bare statement) so these chainable methods'
# return values aren't treated as separate top-level expression outputs.
# Quarto dashboards set Jupyter's shell interactivity to "all" so every
# top-level expression is displayed - since Plotly's add_trace/update_layout
# return `self`, calling them as bare statements was producing extra charts.
fig = fig.add_trace(go.Scatter(
    x=x_line,
    y=slope * x_line + intercept,
    mode='lines',
    name=f'y = {slope:.2f}x + {intercept:.2f}',
    line=dict(color='red'),
))
fig = fig.update_layout(
    xaxis_title='FAR',
    yaxis_title='AU / Acre',
    legend_title_text='',
)
fig

# FAR Table

In [15]:
#| title: "FAR by Growth Center"
au[['county','growth_center','gross_sqft_non_res', 'gross_sqft_res', 'parcel_sqft_non_res',
       'parcel_sqft_res', 'gross_sqft_all', 'parcel_sqft_all', 'far_res',
       'far_non_res', 'far_all']]

,county,growth_center,gross_sqft_non_res,gross_sqft_res,parcel_sqft_non_res,parcel_sqft_res,gross_sqft_all,parcel_sqft_all,far_res,far_non_res,far_all
0,King,Auburn,3326850.0,1768626.0,6831324,3548514,5095476.0,10379837,0.498413,0.486999,0.490901
1,King,Bellevue,40617329.0,10636173.0,12069664,1090863,51253502.0,13160528,9.750237,3.365241,3.894487
2,King,Burien,3381113.0,1806127.0,9284369,3473620,5187240.0,12757990,0.519955,0.364173,0.406588
3,King,Federal Way,2494599.0,261635.0,7932634,106449,2756234.0,8039083,2.457844,0.314473,0.342854
4,King,Kent,2994430.0,1186542.0,7282803,855229,4180972.0,8138032,1.387397,0.411164,0.513757
5,King,Kirkland Totem Lake,7743882.0,5871758.0,17611638,7489062,13615640.0,25100700,0.784045,0.439703,0.542441
6,King,Redmond Downtown,5805982.0,8424325.0,11522679,1789425,14230307.0,13312104,4.707839,0.503874,1.068975
7,King,Redmond-Overlake,24173230.0,5922963.0,26884314,2873960,30096193.0,29758275,2.060907,0.899157,1.011355
8,King,Renton,10509394.0,3814252.0,16018733,2576038,14323646.0,18594771,1.480666,0.656069,0.770305
9,King,SeaTac,9012023.0,4138448.0,15438639,10269541,13150471.0,25708181,0.402983,0.583732,0.511529


# Activity Units Table

In [16]:
#| title: "Activity Units by Growth Center"
au[['county','growth_center','jobs','persons','activity_units','acres_land','au_acre']]

,county,growth_center,jobs,persons,activity_units,acres_land,au_acre
0,King,Auburn,5354,3612.0,8966.0,238.288269,37.626695
1,King,Bellevue,52050,16544.0,68594.0,302.124151,227.039116
2,King,Burien,3758,3851.0,7609.0,292.883150,25.979644
3,King,Federal Way,2737,604.0,3341.0,184.551951,18.103304
4,King,Kent,6091,2153.0,8244.0,186.823508,44.127209
5,King,Kirkland Totem Lake,15575,8227.0,23802.0,576.232782,41.306223
6,King,Redmond Downtown,11142,9678.0,20820.0,305.603857,68.127412
7,King,Redmond-Overlake,60566,7002.0,67568.0,683.155992,98.905668
8,King,Renton,16601,5480.0,22081.0,426.877204,51.726819
9,King,SeaTac,26970,11483.0,38453.0,590.178627,65.154850


# Notes

- data source: urbansim 2023_parcel_baseyear data
- Activity units = persons (from households table) + jobs
- home-based jobs are not included
- Activity units per acre = activity units / parcel_sqft / 43560
- FAR is calculated as gross_sqft (from buildings) / parcel_sqft